## Data Ingestion & Structural Audit

In [96]:
import numpy as np 
import pandas as pd

In [97]:
# 1.1 Load the saas_telemetry_raw.csv dataset into a Pandas DataFrame.
df = pd.read_csv("saas_telemetry_raw.csv")

In [98]:
# 1.2 Perform a structural audit to understand the dataset: Shape, Data types, Missing values
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 300 entries, 0 to 299
Data columns (total 6 columns):
 #   Column               Non-Null Count  Dtype
---  ------               --------------  -----
 0   user_id              300 non-null    str  
 1   signup_date          300 non-null    str  
 2   region               300 non-null    str  
 3   subscription_tier    300 non-null    str  
 4   monthly_usage_hours  298 non-null    str  
 5   monthly_bill         299 non-null    str  
dtypes: str(6)
memory usage: 14.2 KB


In [99]:
# 1.3 Identify columns containing non-standard missing value flags such as: "N/A", "missing", "-999"
df.isna().sum()

user_id                0
signup_date            0
region                 0
subscription_tier      0
monthly_usage_hours    2
monthly_bill           1
dtype: int64

In [100]:
df['monthly_bill'] = pd.to_numeric(df['monthly_bill'], errors='coerce')
df['monthly_usage_hours'] = pd.to_numeric(df['monthly_usage_hours'], errors='coerce')

In [101]:
(df['monthly_bill']<0).any()

np.False_

In [102]:
(df['monthly_usage_hours']<0).any()

np.True_

## Task 2: Data Cleaning & Quality Enforcement

In [103]:
# 2.1 Standardize Null Values
# Replace all non-standard missing value indicators:
    # "N/A"
    # "missing"
    # "-999"

df = df.replace(["N/A", "missing", "-999", -999], np.nan)

In [104]:
# 2.2 Convert the signup_date column to the datetime64 data type.
df['signup_date'] = pd.to_datetime(df['signup_date'])

In [116]:
# 2.3 Handle Data Anomalies
# The monthly_usage_hours column contains system bugs where 
    # negative usage hours were recorded.
# Convert any negative values in this column to np.nan.

df.loc[df['monthly_usage_hours']<0,'monthly_usage_hours'] = np.nan

In [106]:
(df['monthly_usage_hours'] < 0).any()

np.False_

In [107]:
df.isna().sum()

user_id                0
signup_date            0
region                 0
subscription_tier      0
monthly_usage_hours    6
monthly_bill           2
dtype: int64

In [108]:
df['monthly_bill'] = df['monthly_bill'].fillna(df['monthly_bill'].median())

In [109]:
df.isna().sum()

user_id                0
signup_date            0
region                 0
subscription_tier      0
monthly_usage_hours    6
monthly_bill           0
dtype: int64

In [115]:
# 2.4 Impute Missing Values
# Fill missing values in monthly_usage_hours 
    # using the median usage hours of the user's corresponding subscription_tier.
# Hint: Use groupby() together with transform().

df['monthly_usage_hours'] = df['monthly_usage_hours'].fillna(
    df.groupby('subscription_tier')['monthly_usage_hours'].transform('median')
)

In [111]:
df.isna().sum()

user_id                0
signup_date            0
region                 0
subscription_tier      0
monthly_usage_hours    0
monthly_bill           0
dtype: int64

## Task 3: Feature Engineering with Group Transformations

In [139]:
# 3.1 tier_avg_usage
# Compute the average monthly_usage_hours for each user's subscription_tier.

# tier_avg_usage = df.groupby('subscription_tier').agg(
#     tier_avg_usage = ('monthly_usage_hours','mean')
# )

df['tier_avg_usage'] = df.groupby('subscription_tier')['monthly_usage_hours'].transform('mean')

In [143]:
df.groupby('subscription_tier')['tier_avg_usage'].mean()

subscription_tier
Basic          22.910959
Enterprise    232.378261
Pro            73.736111
Name: tier_avg_usage, dtype: float64

In [162]:
# 3.2 usage_deviation
# Calculate the deviation from the subscription tier average:
# usage_deviation = monthly_usage_hours - tier_avg_usage

In [142]:
df['usage_deviation'] = df['monthly_usage_hours'] - df['tier_avg_usage']

In [156]:
df.groupby(['subscription_tier'])['usage_deviation'].mean()

subscription_tier
Basic         8.395111e-16
Enterprise    2.038949e-14
Pro          -3.618505e-15
Name: usage_deviation, dtype: float64

In [161]:
df.groupby('subscription_tier')[['tier_avg_usage', 'usage_deviation']].mean()

,tier_avg_usage,usage_deviation
subscription_tier,,
Basic,22.910959,8.395111e-16
Enterprise,232.378261,2.038949e-14
Pro,73.736111,-3.618505e-15


## Task 4: Executive Aggregations (Split-Apply-Combine)

Create an **executive summary DataFrame** by grouping the data using:
- `subscription_tier`
- `region`

Use **Named Aggregations** with `.agg()` to calculate the following metrics:
| Metric | Description |
|---------|-------------|
| `total_users` | Count of unique `user_id` values |
| `avg_monthly_bill` | Mean of `monthly_bill` |
| `total_usage_hours` | Sum of `monthly_usage_hours` |
| `max_usage_deviation` | Maximum value of `usage_deviation` |

In [168]:
executive_summary = df.groupby(['subscription_tier','region']).agg(
    total_users = ('user_id','nunique'),
    avg_monthly_bill = ('monthly_bill','mean'),
    total_usage_hours = ('monthly_usage_hours','sum'),
    max_usage_deviation =('usage_deviation','max')
)

In [170]:
executive_summary

total_users  avg_monthly_bill  \
subscription_tier region                                         
Basic             APAC                    32         12.347188   
                  EMEA                    37         12.676486   
                  LATAM                   44         12.571818   
                  North America           33         12.915455   
Enterprise        APAC                    11        390.780000   
                  EMEA                    12        315.526667   
                  LATAM                   11        316.434545   
                  North America           12        374.453333   
Pro               APAC                    19         39.588421   
                  EMEA                    28         38.621071   
                  LATAM                   22         41.223182   
                  North America           39         40.774103   

                                 total_usage_hours  max_usage_deviation  
subscription_tier region                                                 
Basic             APAC                      706.05            16.289041  
                  EMEA                      836.70            16.989041  
                  LATAM                     989.90            16.189041  
                  North America             812.35            16.489041  
Enterprise        APAC                     2509.30            87.121739  
                  EMEA                     3104.30           110.221739  
                  LATAM                    2611.80            84.621739  
                  North America            2464.00           105.821739  
Pro               APAC                     1327.70            35.463889  
                  EMEA                     1840.10            25.363889  
                  LATAM                    1610.10            43.763889  
                  North America            3185.60            41.563889

## Task 5: Reshaping for Business Reporting

The executive team requires a **wide-format** report comparing average billing across regions.

Create a **Pivot Table** using `pivot_table()` with the following configuration:

| Parameter | Value |
|-----------|-------|
| **Rows (Index)** | `subscription_tier` |
| **Columns** | `region` |
| **Values** | `monthly_bill` |
| **Aggregation Function** | Mean |
| **Margins** | Include row and column totals (subtotals) |

In [173]:
df.pivot_table(
    index='subscription_tier',
    columns='region',
    values='monthly_bill',
    aggfunc='mean',
    margins=True,
    margins_name="subtotals"
)

region,APAC,EMEA,LATAM,North America,subtotals
subscription_tier,,,,,
Basic,12.347188,12.676486,12.571818,12.915455,12.626781
Enterprise,390.780000,315.526667,316.434545,374.453333,349.111304
Pro,39.588421,38.621071,41.223182,40.774103,40.098796
subtotals,87.836613,69.308312,64.166883,77.498095,74.111000
